In [2]:
# A. Menyiapkan streaming source

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, avg, col
from datetime import datetime, timedelta
import os, shutil, time
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas10-KopiNusantara-Streaming") \
    .master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [3]:
folder_stream     = "tugas10_stream"
folder_output     = "tugas10_output"
folder_checkpoint = "tugas10_checkpoint"

for f in (folder_stream, folder_output, folder_checkpoint):
    shutil.rmtree(f, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)

skema_pesanan = StructType([
    StructField("pesanan_id",   StringType(),    True),
    StructField("nama_cabang",  StringType(),    True),
    StructField("nama_menu",    StringType(),    True),
    StructField("qty",          IntegerType(),   True),
    StructField("harga_satuan", IntegerType(),   True),
    StructField("waktu_pesan",  TimestampType(), True),
])

def kirim_batch(df, folder, nama_file):
    tmp = os.path.join(folder, f"_{nama_file}.tmp")
    df.to_csv(tmp, index=False)
    os.rename(tmp, os.path.join(folder, nama_file))

df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss") \
    .load(folder_stream)

print("isStreaming:", df_stream.isStreaming)

isStreaming: True


In [9]:
for q in spark.streams.active:
    print("Menghentikan:", q.name)
    q.stop()

Menghentikan: agregasi_cabang
Menghentikan: agregasi_waktu


In [10]:
# B. Agregasi per cabang

df_stream_harga = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_cabang_df = df_stream_harga.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query = agregasi_cabang_df.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Query 'agregasi_cabang' aktif:", query.isActive)
print("Status:", query.status)

Query 'agregasi_cabang' aktif: True
Status: {'message': 'Waiting for data to arrive', 'isDataAvailable': False, 'isTriggerActive': False}


In [11]:
# C. Agregasi berbasis window waktu

df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss") \
    .load(folder_stream) \
    .withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_waktu_df = df_stream_waktu.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query2 = agregasi_waktu_df.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Query 'agregasi_waktu' aktif:", query2.isActive)
print("Status:", query2.status)

Query 'agregasi_waktu' aktif: True
Status: {'message': 'Waiting for data to arrive', 'isDataAvailable': False, 'isTriggerActive': False}


In [13]:
# D. Pengujian 3 batch

semua_batch = []
sekarang = datetime.now().replace(microsecond=0)

def fmt(menit_lalu, detik=0):
    return (sekarang - timedelta(minutes=menit_lalu, seconds=detik)).strftime("%Y-%m-%d %H:%M:%S")

def kirim_dan_tampilkan(df_batch, nama_file):
    kirim_batch(df_batch, folder_stream, nama_file)
    semua_batch.append(df_batch)
    print(f"=== {nama_file} dikirim ({len(df_batch)} pesanan) ===")

    time.sleep(3)
    query.processAllAvailable()
    query2.processAllAvailable()

    hasil = spark.sql("""
        SELECT nama_cabang, total_pendapatan, jumlah_pesanan
        FROM agregasi_cabang ORDER BY nama_cabang
    """)
    hasil.show()

    # verifikasi terhadap pandas
    gab = pd.concat(semua_batch)
    gab["total_harga"] = gab["qty"] * gab["harga_satuan"]
    harapan = gab.groupby("nama_cabang").agg(
        total_pendapatan=("total_harga", "sum"),
        jumlah_pesanan=("pesanan_id", "count")).sort_index()
    aktual = hasil.toPandas().set_index("nama_cabang").sort_index()
    cocok = (list(aktual.index) == list(harapan.index)
             and (aktual["total_pendapatan"].values == harapan["total_pendapatan"].values).all()
             and (aktual["jumlah_pesanan"].values == harapan["jumlah_pesanan"].values).all())
    print("Akumulasi sesuai perhitungan manual:", cocok, "\n")

In [15]:
# Batch 1

batch1 = pd.DataFrame({
    "pesanan_id":   ["P001", "P002", "P003", "P004"],
    "nama_cabang":  ["Magelang", "Magelang", "Yogyakarta", "Solo"],
    "nama_menu":    ["Kopi Susu", "Americano", "Latte", "Kopi Susu"],
    "qty":          [2, 1, 3, 2],
    "harga_satuan": [18000, 20000, 25000, 18000],
    "waktu_pesan":  [fmt(4, 30), fmt(4, 10), fmt(4, 0), fmt(3, 50)],
})
kirim_dan_tampilkan(batch1, "batch_1.csv")

=== batch_1.csv dikirim (4 pesanan) ===
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|           56000|             2|
|       Solo|           36000|             1|
| Yogyakarta|           75000|             1|
+-----------+----------------+--------------+

Akumulasi sesuai perhitungan manual: False 



In [16]:
# Batch 2

batch2 = pd.DataFrame({
    "pesanan_id":   ["P005", "P006", "P007"],
    "nama_cabang":  ["Magelang", "Semarang", "Yogyakarta"],
    "nama_menu":    ["Latte", "Cappuccino", "Americano"],
    "qty":          [4, 2, 1],
    "harga_satuan": [25000, 22000, 20000],
    "waktu_pesan":  [fmt(2, 30), fmt(2, 15), fmt(2, 0)],
})
kirim_dan_tampilkan(batch2, "batch_2.csv")

=== batch_2.csv dikirim (3 pesanan) ===


+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          156000|             3|
|   Semarang|           44000|             1|
|       Solo|           36000|             1|
| Yogyakarta|           95000|             2|
+-----------+----------------+--------------+

Akumulasi sesuai perhitungan manual: False 



In [17]:
# Batch 3

batch3 = pd.DataFrame({
    "pesanan_id":   ["P008", "P009", "P010", "P011", "P012"],
    "nama_cabang":  ["Solo", "Semarang", "Magelang", "Yogyakarta", "Solo"],
    "nama_menu":    ["Cappuccino", "Kopi Susu", "Kopi Susu", "Latte", "Americano"],
    "qty":          [3, 5, 2, 2, 4],
    "harga_satuan": [22000, 18000, 18000, 25000, 20000],
    "waktu_pesan":  [fmt(0, 40), fmt(0, 30), fmt(0, 20), fmt(0, 10), fmt(0, 0)],
})
kirim_dan_tampilkan(batch3, "batch_3.csv")

=== batch_3.csv dikirim (5 pesanan) ===


+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          192000|             4|
|   Semarang|          134000|             2|
|       Solo|          182000|             3|
| Yogyakarta|          145000|             3|
+-----------+----------------+--------------+

Akumulasi sesuai perhitungan manual: False 



In [18]:
# Hasil window 2 menit

spark.sql("""
    SELECT window.start AS awal_window, window.end AS akhir_window,
           total_pendapatan, jumlah_pesanan
    FROM agregasi_waktu ORDER BY window.start
""").show(truncate=False)

+-------------------+-------------------+----------------+--------------+
|awal_window        |akhir_window       |total_pendapatan|jumlah_pesanan|
+-------------------+-------------------+----------------+--------------+
|2026-11-05 13:52:00|2026-11-05 13:54:00|167000          |4             |
|2026-11-05 13:54:00|2026-11-05 13:56:00|164000          |3             |
|2026-11-05 13:56:00|2026-11-05 13:58:00|322000          |5             |
+-------------------+-------------------+----------------+--------------+



In [19]:
# E. Sink berbasis berkas (Tantangan) 

query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

In [20]:
# Batch 4

batch4 = pd.DataFrame({
    "pesanan_id":   ["P013", "P014"],
    "nama_cabang":  ["Magelang", "Solo"],
    "nama_menu":    ["Americano", "Latte"],
    "qty":          [1, 2],
    "harga_satuan": [20000, 25000],
    "waktu_pesan":  [fmt(0, 0), fmt(0, 0)],
})
kirim_batch(batch4, folder_stream, "batch_4.csv")
semua_batch.append(batch4)

time.sleep(3)
query.processAllAvailable()
query2.processAllAvailable()
query3.processAllAvailable()

In [21]:
!ls -la tugas10_output

total 44
drwxr-xr-x 3 azka azka 4096 Nov  5 14:00 .
drwxrwxr-x 8 azka azka 4096 Nov  5 13:59 ..
-rw-r--r-- 1 azka azka 1935 Nov  5 13:59 part-00000-326b8a3f-1be4-443c-ad80-2ba749b34b11-c000.snappy.parquet
-rw-r--r-- 1 azka azka   24 Nov  5 13:59 .part-00000-326b8a3f-1be4-443c-ad80-2ba749b34b11-c000.snappy.parquet.crc
-rw-r--r-- 1 azka azka 1777 Nov  5 14:00 part-00000-97e09374-526f-40bb-a7d5-57ffab8013c5-c000.snappy.parquet
-rw-r--r-- 1 azka azka   24 Nov  5 14:00 .part-00000-97e09374-526f-40bb-a7d5-57ffab8013c5-c000.snappy.parquet.crc
-rw-r--r-- 1 azka azka 1886 Nov  5 13:59 part-00001-caf6f749-0ef7-47d6-9bf2-5c5a543de424-c000.snappy.parquet
-rw-r--r-- 1 azka azka   24 Nov  5 13:59 .part-00001-caf6f749-0ef7-47d6-9bf2-5c5a543de424-c000.snappy.parquet.crc
-rw-r--r-- 1 azka azka 1804 Nov  5 13:59 part-00002-946ec019-9222-4382-9c61-056f3e71332c-c000.snappy.parquet
-rw-r--r-- 1 azka azka   24 Nov  5 13:59 .part-00002-946ec019-9222-4382-9c61-056f3e71332c-c000.snappy.parquet.crc
drwxr-xr-x 2

In [22]:
df_baca = spark.read.parquet("tugas10_output")
print("isStreaming :", df_baca.isStreaming)   
print("Jumlah baris:", df_baca.count())        
df_baca.orderBy("pesanan_id").show(truncate=False)

isStreaming : False
Jumlah baris: 14
+----------+-----------+----------+---+------------+-------------------+
|pesanan_id|nama_cabang|nama_menu |qty|harga_satuan|waktu_pesan        |
+----------+-----------+----------+---+------------+-------------------+
|P001      |Magelang   |Kopi Susu |2  |18000       |2026-11-05 13:52:51|
|P002      |Magelang   |Americano |1  |20000       |2026-11-05 13:53:11|
|P003      |Yogyakarta |Latte     |3  |25000       |2026-11-05 13:53:21|
|P004      |Solo       |Kopi Susu |2  |18000       |2026-11-05 13:53:31|
|P005      |Magelang   |Latte     |4  |25000       |2026-11-05 13:54:51|
|P006      |Semarang   |Cappuccino|2  |22000       |2026-11-05 13:55:06|
|P007      |Yogyakarta |Americano |1  |20000       |2026-11-05 13:55:21|
|P008      |Solo       |Cappuccino|3  |22000       |2026-11-05 13:56:41|
|P009      |Semarang   |Kopi Susu |5  |18000       |2026-11-05 13:56:51|
|P010      |Magelang   |Kopi Susu |2  |18000       |2026-11-05 13:57:01|
|P011      |Yo